<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex12.2-power-grid-stability-prediction/Ex12.2_07_andes_comparison.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->


# Ex_12.2 · Notebook 07 — Checking the Labels Against ANDES

**Supplementary. Nothing in notebooks 00 to 06 depends on it.**

*Contributed by Shahariya Rasheed.*

---

### Why this notebook exists

Every number the models in this set are trained on is a critical clearing time
that `problem.py` computed. The whole exercise — the MAEs, the held-out
topology, the screening thresholds — rests on those labels being right. Nothing
in notebooks 00 to 06 checks them, because there is nothing to check them
against: the simulator and the ground truth are the same code.

So this notebook runs the same network, the same contingencies and the same
dispatches through [ANDES](https://docs.andes.app/), an independent open-source
power system simulator, and compares. That is a different kind of test from
anything else in the course. Every other notebook asks whether a model learned
what the data says. This one asks whether the data is true.

### What the classical model leaves out, and what it costs

`problem.py` makes three simplifications, each deliberate and each documented
where it is made:

| simplification | what ANDES does instead |
|---|---|
| fault as a global weakening of the network | an actual short at the faulted bus |
| constant voltage behind transient reactance | GENROU, with subtransient reactances |
| no exciter, no governor | ESST1A and TGOV1 acting during the swing |

Kron reduction is the other difference. `problem.py` eliminates the load buses
to leave a two-machine system, which is what makes the swing equation small
enough to write on a slide and fast enough to label a thousand cases. ANDES
solves the full differential-algebraic system by trapezoidal integration and
reduces nothing. The cost of that is visible in the timings this notebook
prints, and it is the reason the exercise uses the reduction rather than an
apology for it.

### The question that matters

Not "are the CCTs close". The question is whether any case changes its
**security classification** — where `problem.py` says secure and ANDES says
insecure. A screen trained on optimistic labels is optimistic in operation, and
that is the failure mode L12.2 spends a whole slide on. Section 3 counts them.

If that count is zero or near it, the simplified labels are fit to train on and
you can say so with evidence rather than by assertion. If it is not, that is a
more interesting result than anything else in this set, and the report should
lead with it.


In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py',
         'andes_compare.py', 'andes_dataset.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex12.2-power-grid-stability-prediction/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
!pip install andes -q

In [ ]:
import os
for f in ("course_core.py", "pinn_core.py", "problem.py", "andes_compare.py"):
    if not os.path.exists(f):
        try:
            from google.colab import files
            print(f"Please upload {f}")
            files.upload()
        except ImportError:
            raise FileNotFoundError(f"{f} must sit beside this notebook")

import numpy as np
import matplotlib.pyplot as plt
import problem as pb
import andes_compare as ac

print("problem.py loaded")
print(f"ANDES available: {ac.HAS_ANDES}")
print(f"contingencies: {len(ac.contingencies())}")

---
## 1 · Power Flow Comparison

Same network, same impedances. The difference comes from how the generator
buses are handled: `problem.py` treats Gen east as a PQ bus (voltage free),
ANDES treats it as PV (AVR holds voltage). The slack bus is the same in both.

In [ ]:
V_pb, th_pb, ok, _ = pb.solve_power_flow()
pf = ac.power_flow_comparison(verbose=False)

print(f"  {'Bus':>4}  {'problem.py':>11}  {'ANDES cls':>10}  {'ANDES det':>10}")
for i in range(6):
    print(f"  {i:>4}  {V_pb[i]:>11.4f}  {pf['classical'][i]:>10.4f}  "
          f"{pf['detailed'][i]:>10.4f}")

---
## 2 · One Contingency — Three Models

Trip line 0 (Slack–Gen east) at base dispatch. This is the most severe
contingency — the direct tie between the two generators is lost.

Three CCTs:
- **problem.py** — classical model, `FAULT_ADMITTANCE` at sending-end bus
- **ANDES classical** — GENCLS, same H and D, actual bus fault
- **ANDES detailed** — GENROU + ESST1A + TGOV1, actual bus fault

In [ ]:
import time

op = np.stack(pb.injections())
c = ac.contingencies()[1]  # trip line 0
print(f"Contingency: {c['label']}\n")

# problem.py
t0 = time.time()
cct_pb = pb.label_case(op, c["outage"], c["fault_bus"])
t_pb = time.time() - t0

# ANDES classical
t0 = time.time()
cct_cls = ac.compute_cct(c["fault_bus"], c["outage"], model="classical")
t_cls = time.time() - t0

# ANDES detailed
t0 = time.time()
cct_det = ac.compute_cct(c["fault_bus"], c["outage"], model="detailed")
t_det = time.time() - t0

print(f"  {'Model':<30s}{'CCT (ms)':>10s}{'Time (s)':>10s}{'Secure':>8s}")
print("  " + "─" * 58)
for label, cct, t in [("problem.py (classical)", cct_pb, t_pb),
                       ("ANDES GENCLS", cct_cls, t_cls),
                       ("ANDES GENROU+AVR+Gov", cct_det, t_det)]:
    flag = "yes" if cct > ac.PROTECTION_TIME else "NO"
    print(f"  {label:<30s}{cct*1e3:>8.1f} ms{t:>9.2f} s{flag:>8s}")

---
## 3 · All Contingencies at Base Dispatch

Side-by-side for all six contingencies. Watch which ones change security
classification between models — those are the cases where the simplified
model gives a qualitatively wrong answer.

In [ ]:
results = ac.compare_with_problem(verbose=True)

---
## 4 · Swing Trajectories — Visual Comparison

Same contingency (line 0 trip), same clearing time (100 ms). Compare the
machine response between classical and detailed models.

In [ ]:
t_clear = ac.T_FAULT + 0.10

r_cls = ac.run_case(fault_bus=0, trip_line=0, t_clear=t_clear, model="classical")
r_det = ac.run_case(fault_bus=0, trip_line=0, t_clear=t_clear, model="detailed")

pb.use_course_style()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for r, label, color in [(r_cls, "GENCLS", pb.CYAN),
                         (r_det, "GENROU+AVR+Gov", pb.AMBER)]:
    sep = np.degrees(r["delta"][:, 1] - r["delta"][:, 0])
    freq = r["omega"] * 50
    axes[0].plot(r["t"], sep, lw=1.5, color=color, label=label)
    axes[1].plot(r["t"], freq[:, 1], lw=1.5, color=color, label=label)

axes[0].set_xlabel("t (s)"); axes[0].set_ylabel("angle separation (deg)")
axes[0].legend(fontsize=8); axes[0].set_title("rotor angle separation")
axes[1].set_xlabel("t (s)"); axes[1].set_ylabel("frequency (Hz)")
axes[1].axhline(50, color=pb.MUTED, lw=0.8, ls=":")
axes[1].legend(fontsize=8); axes[1].set_title("Gen east frequency")
fig.tight_layout(); plt.show()

for r, label in [(r_cls, "GENCLS"), (r_det, "GENROU+AVR+Gov")]:
    sep = np.degrees(np.abs(r["delta"][:, 1] - r["delta"][:, 0]))
    freq = r["omega"][:, 1] * 50
    print(f"  {label:<25s}  max sep: {sep.max():.1f} deg"
          f"   freq range: {freq.min():.2f}–{freq.max():.2f} Hz")

---
## 5 · Full Dataset Generation

Generate CCT labels for the same 180 dispatches × 6 contingencies used in
Ex12.2 notebook 01, but using ANDES.

**Start small.** The cell below runs 5 dispatches (30 cases) as a test.
Change `n_ops=5` to `n_ops=180` for the full dataset once you've verified
it works. The full run takes 1–2 hours.

In [ ]:
import andes_dataset as ad

# Quick test — 5 dispatches, 30 cases
data_test = ad.quick_test(n_ops=5, model="classical")

### Full dataset (uncomment to run — takes 1–2 hours)

In [ ]:
# Uncomment the lines below to generate the full 1,080-case dataset.
# Results are cached — the second run loads instantly.

# data = ad.build(n_ops=180, model="classical")
# ad.compare_labels(data)

---
## 6 · What the Comparison Tells You

Three questions this notebook answers:

**Does the fault model matter?**
Compare problem.py vs ANDES classical. Same machine model (constant E, same H).
The difference is entirely in how the fault is applied — global weakening vs
actual bus fault. This is usually the largest difference.

**Does the machine model matter?**
Compare ANDES classical vs ANDES detailed. Same fault model. The difference is
GENROU's subtransient reactances, the exciter recovering voltage, and the
governor adjusting mechanical power. For first-swing stability (CCT), this is
usually a secondary effect.

**Does it change the security classification?**
The dangerous case is: problem.py says secure, ANDES says insecure. A
classifier trained on problem.py labels would miss those cases in operation.
Count them. If the number is large, the simplified labels are not safe to
train on.

These are the numbers to quote in the report — not just the CCT values, but
which model decisions change the answer qualitatively.

---

## 7 · Before you move on

Answer these here. Each question builds part of an answer to one of the lecture's questions for the oral examination; the arrow under it says which, and the Questions slide at the end of the lecture has them in full.

1. `problem.py` uses the classical machine with no exciter and no governor, and labels only first-swing behaviour. Say which of the three stability questions its labels answer, which ANDES's detailed model starts to touch, and what neither model here could say anything about.
   *→ L12.2 Q1*
2. ANDES solves the full differential-algebraic system and reduces nothing. Using the timings section 2 printed, say what the Kron reduction bought this exercise set, and why a thousand labels would have cost far more without it. What does that say about the combinatorial case for a learned screener?
   *→ L12.2 Q2*
3. The first comparison in section 6 changes only how the fault is applied: a global weakening of the network against an actual short at the bus. Using the equal area criterion read as energy, say which area that choice changes and in which direction it would move the critical clearing time.
   *→ L12.2 Q3*
4. Every other notebook asks whether a model learned what the data says; this one asks whether the data is true. Report how many cases changed from secure to insecure between `problem.py` and ANDES, and say what that count means for every model in this set. What does it say about where a physics-informed model's trust actually comes from?
   *→ L12.2 Q10, Q5*


*Write your answers here. You will copy them into the report in notebook 05, which adds them to what you submit.*

1.
2.
3.
4.
